# FHIR Encounter Data Quality Profiling

## Purpose

In this notebook, I profile the Silver FHIR Encounter dataset and define the
data-quality rules that will later be enforced directly inside the Lakeflow
Bronze-to-Silver transformation.

I am not creating another cleaned Encounter table in this notebook.

The purpose of this stage is to:

- identify incomplete or logically invalid Encounter records
- validate required relational identifiers
- inspect optional Practitioner and Organization references
- validate Encounter timing consistency
- inspect Encounter status and class domains
- classify rules as warning, drop, or fail
- prepare reusable Lakeflow expectation definitions

### Source

`health_insurance.silver.fhir_encounter`

### Production design

The final production flow will be:

Bronze  
↓  
FHIR Encounter transformation  
+  
Lakeflow expectations  
↓  
Validated Silver Encounter  
↓  
Gold

### Data-quality approach

FHIR Encounter records contain both required and optional relationships.

I treat the Encounter ID and Patient ID as critical relational fields, while
Practitioner, Organization, and Encounter type completeness are monitored
without automatically rejecting the record.

I also validate that Encounter timestamps remain logically consistent.

In [0]:
# loading the current Silver Encounter dataset for quality profiling.

from pyspark.sql import functions as F

ENCOUNTER_TABLE = "health_insurance.silver.fhir_encounter"

encounter_df = spark.table(ENCOUNTER_TABLE)

print(f"Encounter rows: {encounter_df.count():,}")

encounter_df.printSchema()

display(encounter_df.limit(10))

In [0]:
# defining candidate Encounter quality rules by severity.

ENCOUNTER_WARN_RULES = {
    "practitioner_reference_present":
        "practitioner_id IS NOT NULL",

    "organization_reference_present":
        "organization_id IS NOT NULL",

    "encounter_type_present":
        "encounter_type IS NOT NULL",

    "status_present":
        "status IS NOT NULL",

    "encounter_class_present":
        "encounter_class IS NOT NULL",

    "start_datetime_present":
        "start_datetime IS NOT NULL"
}


ENCOUNTER_DROP_RULES = {
    "encounter_id_present":
        "encounter_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL"
}


ENCOUNTER_FAIL_RULES = {
    "encounter_timeline_valid":
        """
        start_datetime IS NULL
        OR end_datetime IS NULL
        OR end_datetime >= start_datetime
        """
}

In [0]:
# measuring how many Encounter rows violate each candidate quality rule.

def profile_rules(df, rules, severity):

    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(
                f"NOT ({condition}) OR ({condition}) IS NULL"
            )
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition.strip(),
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
# profiling all proposed Encounter quality rules.

encounter_quality_results = []

encounter_quality_results += profile_rules(
    encounter_df,
    ENCOUNTER_WARN_RULES,
    "WARN"
)

encounter_quality_results += profile_rules(
    encounter_df,
    ENCOUNTER_DROP_RULES,
    "DROP"
)

encounter_quality_results += profile_rules(
    encounter_df,
    ENCOUNTER_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the Encounter quality profile as a structured result.

encounter_quality_profile_df = spark.createDataFrame(
    encounter_quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    encounter_quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# inspecting the actual Encounter status and class values
# before finalizing controlled-domain quality rules.

display(
    encounter_df
    .groupBy(
        "status",
        "encounter_class"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

In [0]:
# inspecting Encounter timing and relationship completeness.

encounter_df.select(
    F.count("*").alias("total_encounters"),

    F.sum(
        F.col("start_datetime").isNull().cast("int")
    ).alias("missing_start"),

    F.sum(
        F.col("end_datetime").isNull().cast("int")
    ).alias("missing_end"),

    F.sum(
        (
            F.col("end_datetime")
            < F.col("start_datetime")
        ).cast("int")
    ).alias("end_before_start"),

    F.sum(
        F.col("practitioner_id").isNull().cast("int")
    ).alias("missing_practitioner"),

    F.sum(
        F.col("organization_id").isNull().cast("int")
    ).alias("missing_organization"),

    F.sum(
        F.col("encounter_type").isNull().cast("int")
    ).alias("missing_encounter_type")
).show()

In [0]:
# defining the finalized Encounter quality contract.

ENCOUNTER_WARN_RULES = {
    "practitioner_reference_present":
        "practitioner_id IS NOT NULL",

    "organization_reference_present":
        "organization_id IS NOT NULL",

    "encounter_type_present":
        "encounter_type IS NOT NULL",

    "recognized_status":
        "status IN ('FINISHED')",

    "recognized_encounter_class":
        "encounter_class IN ('AMB', 'EMER', 'IMP')",

    "start_datetime_present":
        "start_datetime IS NOT NULL"
}


ENCOUNTER_DROP_RULES = {
    "encounter_id_present":
        "encounter_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL"
}


ENCOUNTER_FAIL_RULES = {
    "encounter_timeline_valid":
        """
        start_datetime IS NULL
        OR end_datetime IS NULL
        OR end_datetime >= start_datetime
        """
}

In [0]:
# preparing the finalized Encounter quality rules
# for storage in my reusable quality-rules module.

encounter_quality_code = '''
# === ENCOUNTER QUALITY RULES START ===

ENCOUNTER_WARN_RULES = {
    "practitioner_reference_present":
        "practitioner_id IS NOT NULL",

    "organization_reference_present":
        "organization_id IS NOT NULL",

    "encounter_type_present":
        "encounter_type IS NOT NULL",

    "recognized_status":
        "status IN ('FINISHED')",

    "recognized_encounter_class":
        "encounter_class IN ('AMB', 'EMER', 'IMP')",

    "start_datetime_present":
        "start_datetime IS NOT NULL"
}


ENCOUNTER_DROP_RULES = {
    "encounter_id_present":
        "encounter_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL"
}


ENCOUNTER_FAIL_RULES = {
    "encounter_timeline_valid":
        """
        start_datetime IS NULL
        OR end_datetime IS NULL
        OR end_datetime >= start_datetime
        """
}

# === ENCOUNTER QUALITY RULES END ===
'''

In [0]:
# adding or updating the Encounter section
# without changing my existing quality contracts.

from pathlib import Path
import re

QUALITY_RULES_PATH = (
    "/Workspace/Khaoula healthy insurance project/"
    "Khaoula-healthy-insuarance-project/"
    "04-data-quality/"
    "quality_rules.py"
)

quality_file = Path(QUALITY_RULES_PATH)

existing_code = quality_file.read_text(
    encoding="utf-8"
)

start_marker = "# === ENCOUNTER QUALITY RULES START ==="
end_marker = "# === ENCOUNTER QUALITY RULES END ==="

pattern = (
    re.escape(start_marker)
    + r".*?"
    + re.escape(end_marker)
)

if start_marker in existing_code:
    updated_code = re.sub(
        pattern,
        encounter_quality_code.strip(),
        existing_code,
        flags=re.DOTALL
    )
else:
    updated_code = (
        existing_code.rstrip()
        + "\n\n\n"
        + encounter_quality_code.strip()
        + "\n"
    )

quality_file.write_text(
    updated_code,
    encoding="utf-8"
)

print("Encounter quality rules saved successfully.")